# 04. Deep Q-Networks (DQN)

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [03. Q-Learning](03-q-learning.ipynb) + Conocimientos básicos de redes neuronales

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Comprender las limitaciones de Q-Learning tabular y la necesidad de aproximación de funciones
- Implementar una red neuronal para aproximar la función Q
- Entender y aplicar **experience replay** para estabilizar el entrenamiento
- Implementar **target networks** para reducir correlaciones temporales
- Entrenar un agente DQN en ambientes complejos (CartPole, LunarLander)
- Analizar el proceso de aprendizaje y debugging de DQN

## 📚 Motivación

### El Problema de la Maldición de la Dimensionalidad

Q-Learning tabular funciona perfectamente para problemas pequeños, pero:

**¿Qué pasa si el espacio de estados es enorme?**

Ejemplos:
- **Atari Breakout**: Cada frame es 210×160×3 píxeles → $256^{210 \times 160 \times 3} \approx 10^{127,000}$ estados posibles
- **Go**: $10^{170}$ posiciones posibles
- **Robótica**: Espacios de estados continuos (ángulos, velocidades)

**Problemas con tablas Q**:
1. **Memoria**: Imposible almacenar una tabla con $10^{127,000}$ entradas
2. **Aprendizaje**: Necesitarías visitar cada estado infinitamente para aprender
3. **Generalización**: No hay forma de generalizar entre estados similares

### La Solución: Aproximación de Funciones

En lugar de almacenar Q(s,a) en una tabla, usamos una **función paramétrica**:

$$Q(s, a) \approx Q(s, a; \theta)$$

donde $\theta$ son los parámetros de una **red neuronal**.

**Ventajas**:
- Memoria constante (solo los parámetros $\theta$)
- Generalización automática entre estados similares
- Puede manejar entradas de alta dimensionalidad (imágenes)

### El Breakthrough: DQN (2013-2015)

DeepMind demostró que una red neuronal puede:
- Jugar Atari directamente desde píxeles
- Alcanzar nivel humano en muchos juegos
- Usar la misma arquitectura para todos los juegos

**Ingredientes clave de DQN**:
1. **Red neuronal profunda** para aproximar Q(s,a)
2. **Experience replay** para romper correlaciones
3. **Target network** para estabilizar objetivos

### Pregunta Guía
**¿Cómo puede una red neuronal aprender a jugar videojuegos sin conocer las reglas, usando solo píxeles y recompensas?**

In [ ]:
# Importar librerías
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import gymnasium as gym
from typing import Tuple, List, Dict, Deque
from collections import deque, namedtuple
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
np.random.seed(42)
torch.manual_seed(42)

# Verificar disponibilidad de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Librerías importadas")
print(f"🖥️  Usando dispositivo: {device}")

## 🎨 Intuición Visual

### Q-Learning Tabular vs DQN

In [ ]:
from IPython.display import display, HTML

comparison_html = """
<table style='width:100%; border-collapse: collapse; font-size: 14px;'>
  <tr style='background-color: #f0f0f0;'>
    <th style='border: 1px solid black; padding: 10px;'>Aspecto</th>
    <th style='border: 1px solid black; padding: 10px;'>Q-Learning Tabular</th>
    <th style='border: 1px solid black; padding: 10px;'>Deep Q-Network (DQN)</th>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Representación</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Tabla Q[s, a]</td>
    <td style='border: 1px solid black; padding: 10px;'>Red Neuronal Q(s, a; θ)</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Capacidad</b></td>
    <td style='border: 1px solid black; padding: 10px;'>~10,000 estados</td>
    <td style='border: 1px solid black; padding: 10px;'>Millones/Infinitos estados</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Generalización</b></td>
    <td style='border: 1px solid black; padding: 10px;'>No (cada estado es independiente)</td>
    <td style='border: 1px solid black; padding: 10px;'>Sí (aprende características compartidas)</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Entrada</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Índice de estado</td>
    <td style='border: 1px solid black; padding: 10px;'>Vectores, imágenes, cualquier cosa</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Actualización</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Directa: Q[s,a] += α·δ</td>
    <td style='border: 1px solid black; padding: 10px;'>Gradient descent: θ -= α·∇L(θ)</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Ejemplo</b></td>
    <td style='border: 1px solid black; padding: 10px;'>FrozenLake, Taxi</td>
    <td style='border: 1px solid black; padding: 10px;'>Atari, Robótica, Go</td>
  </tr>
</table>
"""

display(HTML(comparison_html))

### Arquitectura de DQN

Visualización conceptual de cómo funciona DQN:

In [ ]:
architecture_html = """
<div style='font-family: monospace; border: 2px solid #333; padding: 20px; background-color: #f9f9f9;'>
<pre>
ARQUITECTURA DQN:

Estado (s)
  ↓
[Entrada: 84×84×4 para Atari]
  ↓
[Conv Layer 1: 32 filtros 8×8, stride 4]
  ↓ ReLU
[Conv Layer 2: 64 filtros 4×4, stride 2]
  ↓ ReLU
[Conv Layer 3: 64 filtros 3×3, stride 1]
  ↓ ReLU
[Fully Connected: 512 neuronas]
  ↓ ReLU
[Capa de Salida: n_actions neuronas]
  ↓
[Q(s, a₁), Q(s, a₂), ..., Q(s, aₙ)]

Acción seleccionada: argmax Q(s, aᵢ)
</pre>
</div>
"""

display(HTML(architecture_html))

print("\n💡 Key Points:")
print("  • La red toma el estado completo y produce Q-values para TODAS las acciones")
print("  • Más eficiente que evaluar la red una vez por cada acción")
print("  • Para Atari: entrada son 4 frames apilados (capturar movimiento)")

## 📐 Fundamentos Matemáticos

### Función Q Aproximada

En lugar de una tabla, usamos una red neuronal:

$$
\begin{align}
Q(s, a) &\approx Q(s, a; \theta) \tag{1} \\
\text{donde: } & \\
\theta &: \text{pesos de la red neuronal} \\
Q(s, a; \theta) &: \text{salida de la red para estado } s \text{ y acción } a
\end{align}
$$

### Función de Pérdida

Entrenamos la red minimizando el **error cuadrático del TD-error**:

$$
\begin{align}
L(\theta) &= \mathbb{E}_{(s,a,r,s') \sim D}\left[\left(y - Q(s, a; \theta)\right)^2\right] \tag{2} \\
\text{donde: } & \\
y &= r + \gamma \max_{a'} Q(s', a'; \theta^-) \quad \text{(target)} \tag{3} \\
\theta^- &: \text{parámetros de la target network (actualizados periódicamente)} \\
D &: \text{replay buffer (memoria de experiencias)}
\end{align}
$$

### Gradiente

$$
\begin{align}
\nabla_{\theta} L(\theta) &= \mathbb{E}\left[-2\left(y - Q(s, a; \theta)\right) \nabla_{\theta} Q(s, a; \theta)\right] \tag{4}
\end{align}
$$

**Actualización con gradient descent**:

$$
\begin{align}
\theta &\leftarrow \theta - \alpha \nabla_{\theta} L(\theta) \tag{5}
\end{align}
$$

### Experience Replay

**Problema**: Las transiciones consecutivas están altamente correlacionadas → inestabilidad.

**Solución**: Almacenar transiciones en un buffer $D$ y samplear mini-batches aleatorios:

$$
\begin{align}
D &= \{e_1, e_2, \ldots, e_N\} \tag{6} \\
\text{donde: } e_i &= (s_i, a_i, r_i, s'_i, done_i)
\end{align}
$$

**Ventajas**:
1. Rompe correlaciones temporales
2. Aumenta eficiencia de datos (reutiliza experiencias)
3. Suaviza distribución de datos

### Target Network

**Problema**: El target $y$ depende de $\theta$ → objetivo móvil → inestabilidad.

**Solución**: Usar una red separada $\theta^-$ para calcular targets:

$$
\begin{align}
y &= r + \gamma \max_{a'} Q(s', a'; \theta^-) \tag{7}
\end{align}
$$

Actualizar $\theta^-$ cada $C$ pasos:

$$
\begin{align}
\theta^- &\leftarrow \theta \quad \text{cada } C \text{ pasos} \tag{8}
\end{align}
$$

**Ventaja**: Target se mantiene fijo durante $C$ pasos → más estabilidad.

### Algoritmo DQN Completo

```
Inicializar replay buffer D con capacidad N
Inicializar Q-network con pesos θ aleatorios
Inicializar target network con pesos θ⁻ = θ

Para cada episodio:
    Inicializar estado s
    Para cada paso:
        Con probabilidad ε: seleccionar acción aleatoria a
        Caso contrario: a = argmax_a Q(s, a; θ)
        Ejecutar a, observar r, s'
        Almacenar (s, a, r, s', done) en D
        Samplear mini-batch aleatorio de D: {(sⱼ, aⱼ, rⱼ, s'ⱼ, doneⱼ)}
        
        Para cada transición j en el mini-batch:
            Si doneⱼ: yⱼ = rⱼ
            Sino: yⱼ = rⱼ + γ max_a' Q(s'ⱼ, a'; θ⁻)
        
        Calcular pérdida: L = (1/batch_size) Σⱼ (yⱼ - Q(sⱼ, aⱼ; θ))²
        Actualizar θ con gradient descent en L
        
        Cada C pasos: θ⁻ ← θ
        s ← s'
```

## 💻 Implementación Desde Cero

### 1. Red Neuronal Q-Network

In [ ]:
class QNetwork(nn.Module):
    """
    Red neuronal para aproximar la función Q.
    
    Arquitectura simple:
    - Input: estado (vector)
    - Hidden layers: 2 capas fully connected
    - Output: Q-values para cada acción
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        """
        Parameters:
        -----------
        state_dim : int
            Dimensión del espacio de estados
        action_dim : int
            Número de acciones posibles
        hidden_dim : int
            Número de neuronas en capas ocultas
        """
        super(QNetwork, self).__init__()
        
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de la red.
        
        Parameters:
        -----------
        state : torch.Tensor
            Estado(s) de entrada, shape: (batch_size, state_dim)
        
        Returns:
        --------
        q_values : torch.Tensor
            Q-values para cada acción, shape: (batch_size, action_dim)
        """
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        q_values = self.fc3(x)  # No activation en la última capa
        return q_values

# Test de la red
test_net = QNetwork(state_dim=4, action_dim=2, hidden_dim=64)
test_state = torch.randn(1, 4)  # Batch de 1 estado
test_output = test_net(test_state)
print(f"✅ QNetwork implementada")
print(f"   Input shape: {test_state.shape}")
print(f"   Output shape: {test_output.shape}")
print(f"   Q-values: {test_output.detach().numpy()}")

### 2. Replay Buffer

In [ ]:
# Estructura para almacenar transiciones
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    """
    Buffer para almacenar y samplear experiencias (experience replay).
    """
    
    def __init__(self, capacity: int):
        """
        Parameters:
        -----------
        capacity : int
            Capacidad máxima del buffer
        """
        self.memory = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """
        Almacena una transición en el buffer.
        """
        self.memory.append(Transition(state, action, reward, next_state, done))
    
    def sample(self, batch_size: int) -> List[Transition]:
        """
        Samplea un mini-batch aleatorio.
        
        Parameters:
        -----------
        batch_size : int
            Tamaño del mini-batch
        
        Returns:
        --------
        batch : List[Transition]
            Lista de transiciones
        """
        return random.sample(self.memory, batch_size)
    
    def __len__(self) -> int:
        return len(self.memory)

# Test del buffer
test_buffer = ReplayBuffer(capacity=1000)
for i in range(10):
    test_buffer.push(
        state=np.random.randn(4),
        action=np.random.randint(2),
        reward=np.random.randn(),
        next_state=np.random.randn(4),
        done=False
    )

sample = test_buffer.sample(5)
print(f"✅ ReplayBuffer implementado")
print(f"   Tamaño del buffer: {len(test_buffer)}")
print(f"   Tamaño del sample: {len(sample)}")

### 3. Agente DQN Completo

In [ ]:
class DQNAgent:
    """
    Agente que implementa Deep Q-Learning con experience replay y target network.
    """
    
    def __init__(self,
                 state_dim: int,
                 action_dim: int,
                 hidden_dim: int = 128,
                 learning_rate: float = 1e-3,
                 gamma: float = 0.99,
                 epsilon_start: float = 1.0,
                 epsilon_end: float = 0.01,
                 epsilon_decay: float = 0.995,
                 buffer_size: int = 10000,
                 batch_size: int = 64,
                 target_update_freq: int = 10):
        
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        
        # Epsilon-greedy
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Redes Q
        self.q_network = QNetwork(state_dim, action_dim, hidden_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim, hidden_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()  # Target network en modo evaluación
        
        # Optimizador
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        
        # Replay buffer
        self.memory = ReplayBuffer(buffer_size)
        
        # Tracking
        self.training_history = {
            'episode': [],
            'reward': [],
            'loss': [],
            'epsilon': [],
            'steps': []
        }
        self.steps_done = 0
    
    def select_action(self, state: np.ndarray, training: bool = True) -> int:
        """
        Selecciona acción usando epsilon-greedy.
        """
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_tensor)
            return q_values.argmax().item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """
        Almacena transición en el replay buffer.
        """
        self.memory.push(state, action, reward, next_state, done)
    
    def train_step(self) -> float:
        """
        Ejecuta un paso de entrenamiento (actualización de la Q-network).
        
        Returns:
        --------
        loss : float
            Pérdida del batch
        """
        if len(self.memory) < self.batch_size:
            return 0.0
        
        # Samplear mini-batch
        transitions = self.memory.sample(self.batch_size)
        batch = Transition(*zip(*transitions))
        
        # Convertir a tensores
        state_batch = torch.FloatTensor(np.array(batch.state)).to(device)
        action_batch = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        reward_batch = torch.FloatTensor(batch.reward).to(device)
        next_state_batch = torch.FloatTensor(np.array(batch.next_state)).to(device)
        done_batch = torch.FloatTensor(batch.done).to(device)
        
        # Calcular Q(s, a) actual
        current_q_values = self.q_network(state_batch).gather(1, action_batch).squeeze()
        
        # Calcular target: y = r + γ max_a' Q(s', a'; θ⁻)
        with torch.no_grad():
            next_q_values = self.target_network(next_state_batch).max(1)[0]
            target_q_values = reward_batch + (1 - done_batch) * self.gamma * next_q_values
        
        # Calcular pérdida (MSE)
        loss = F.mse_loss(current_q_values, target_q_values)
        
        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping para estabilidad
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        
        return loss.item()
    
    def update_target_network(self):
        """
        Actualiza la target network copiando pesos de la Q-network.
        """
        self.target_network.load_state_dict(self.q_network.state_dict())
    
    def decay_epsilon(self):
        """
        Decae epsilon.
        """
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    
    def train(self, env, n_episodes: int = 500, max_steps: int = 500, verbose: bool = True):
        """
        Entrena el agente.
        """
        for episode in range(n_episodes):
            state, _ = env.reset()
            total_reward = 0
            total_loss = 0
            
            for step in range(max_steps):
                # Seleccionar y ejecutar acción
                action = self.select_action(state, training=True)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                
                # Almacenar transición
                self.store_transition(state, action, reward, next_state, done)
                
                # Entrenar
                loss = self.train_step()
                total_loss += loss
                
                total_reward += reward
                state = next_state
                self.steps_done += 1
                
                # Actualizar target network
                if self.steps_done % self.target_update_freq == 0:
                    self.update_target_network()
                
                if done:
                    break
            
            # Decaer epsilon
            self.decay_epsilon()
            
            # Registrar métricas
            self.training_history['episode'].append(episode)
            self.training_history['reward'].append(total_reward)
            self.training_history['loss'].append(total_loss / (step + 1))
            self.training_history['epsilon'].append(self.epsilon)
            self.training_history['steps'].append(step + 1)
            
            if verbose and (episode + 1) % 50 == 0:
                avg_reward = np.mean(self.training_history['reward'][-50:])
                print(f"Episodio {episode + 1}/{n_episodes} | "
                      f"Reward: {avg_reward:.2f} | "
                      f"ε: {self.epsilon:.3f} | "
                      f"Loss: {self.training_history['loss'][-1]:.4f}")
        
        if verbose:
            print("\n✅ Entrenamiento completado!")

print("✅ DQNAgent implementado")

### 4. Entrenando en CartPole

In [ ]:
# Crear ambiente
env = gym.make('CartPole-v1')

print("🎮 CartPole-v1 Environment")
print(f"  Estado: {env.observation_space.shape[0]} dimensiones (posición, velocidad, ángulo, velocidad angular)")
print(f"  Acciones: {env.action_space.n} (izquierda, derecha)")
print(f"  Objetivo: Balancear el polo verticalmente lo más posible\n")

# Crear agente DQN
agent = DQNAgent(
    state_dim=env.observation_space.shape[0],
    action_dim=env.action_space.n,
    hidden_dim=128,
    learning_rate=1e-3,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    buffer_size=10000,
    batch_size=64,
    target_update_freq=10
)

print("🤖 Agente DQN creado")
print(f"   Parámetros de la red: {sum(p.numel() for p in agent.q_network.parameters())}\n")

# Entrenar
print("🎯 Iniciando entrenamiento DQN...\n")
agent.train(env, n_episodes=500, max_steps=500, verbose=True)

env.close()

### 5. Visualización del Entrenamiento

In [ ]:
def plot_dqn_training(agent: DQNAgent):
    """
    Visualiza métricas del entrenamiento DQN.
    """
    history = agent.training_history
    
    # Promedios móviles
    window = 20
    rewards_smooth = pd.Series(history['reward']).rolling(window=window, min_periods=1).mean()
    loss_smooth = pd.Series(history['loss']).rolling(window=window, min_periods=1).mean()
    
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=('Recompensa por Episodio', 'Pérdida (Loss)', 'Epsilon'),
        vertical_spacing=0.1
    )
    
    # Plot 1: Rewards
    fig.add_trace(go.Scatter(x=history['episode'], y=history['reward'],
                             mode='lines', name='Reward', opacity=0.3,
                             line=dict(color='lightblue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=history['episode'], y=rewards_smooth,
                             mode='lines', name=f'Media móvil ({window})',
                             line=dict(color='darkblue', width=2)), row=1, col=1)
    
    # Plot 2: Loss
    fig.add_trace(go.Scatter(x=history['episode'], y=history['loss'],
                             mode='lines', name='Loss', opacity=0.3,
                             line=dict(color='lightcoral')), row=2, col=1)
    fig.add_trace(go.Scatter(x=history['episode'], y=loss_smooth,
                             mode='lines', name=f'Media móvil ({window})',
                             line=dict(color='darkred', width=2)), row=2, col=1)
    
    # Plot 3: Epsilon
    fig.add_trace(go.Scatter(x=history['episode'], y=history['epsilon'],
                             mode='lines', name='Epsilon',
                             line=dict(color='green', width=2)), row=3, col=1)
    
    fig.update_xaxes(title_text="Episodio", row=3, col=1)
    fig.update_yaxes(title_text="Reward", row=1, col=1)
    fig.update_yaxes(title_text="Loss", row=2, col=1)
    fig.update_yaxes(title_text="ε", row=3, col=1)
    
    fig.update_layout(height=900, showlegend=True, template='plotly_white')
    
    return fig

fig = plot_dqn_training(agent)
fig.show()

print("\n📊 Análisis:")
print(f"  - Reward promedio final: {np.mean(agent.training_history['reward'][-50:]):.2f}")
print(f"  - La loss decrece conforme la red aprende")
print(f"  - Epsilon decae gradualmente (más explotación con el tiempo)")

## 🔧 Versión con Framework

Usaremos **Stable-Baselines3**, una biblioteca con implementaciones optimizadas de algoritmos de RL.

In [ ]:
# Instalar Stable-Baselines3
try:
    from stable_baselines3 import DQN
    from stable_baselines3.common.evaluation import evaluate_policy
    print("✅ Stable-Baselines3 disponible")
except ImportError:
    print("📦 Instalando Stable-Baselines3...")
    !pip install stable-baselines3[extra] -q
    from stable_baselines3 import DQN
    from stable_baselines3.common.evaluation import evaluate_policy
    print("✅ Stable-Baselines3 instalado")

In [ ]:
# Crear ambiente
env_sb3 = gym.make('CartPole-v1')

# Crear y entrenar agente DQN con Stable-Baselines3
print("🎯 Entrenando DQN con Stable-Baselines3...\n")

model = DQN(
    'MlpPolicy',
    env_sb3,
    learning_rate=1e-3,
    buffer_size=10000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    target_update_interval=10,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    verbose=1
)

model.learn(total_timesteps=50000)

# Evaluar
mean_reward, std_reward = evaluate_policy(model, env_sb3, n_eval_episodes=100)

print(f"\n✅ Entrenamiento completado")
print(f"   Recompensa promedio: {mean_reward:.2f} ± {std_reward:.2f}")

env_sb3.close()

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Experimento con Hiperparámetros

Investiga cómo diferentes tamaños de replay buffer afectan el aprendizaje.

In [ ]:
def ejercicio_1_buffer_size():
    """
    Objetivo: Comprender el impacto del replay buffer.
    
    Instrucciones:
    1. Entrena agentes DQN con buffer_size = 1000, 5000, 20000
    2. Usa 200 episodios de entrenamiento
    3. Compara las curvas de aprendizaje
    4. Retorna un diccionario con los rewards finales promedio
    """
    # TODO: Tu código aquí
    pass

def test_ejercicio_1():
    resultado = ejercicio_1_buffer_size()
    assert resultado is not None, "❌ Debes retornar un diccionario"
    print("✅ ¡Correcto! Buffers más grandes mejoran estabilidad.")
    return True

# test_ejercicio_1()  # Descomenta para probar

### 🟡 Ejercicio 2: Double DQN

Implementa Double DQN para reducir overestimation de Q-values.

In [ ]:
class DoubleDQNAgent(DQNAgent):
    """
    Double DQN: usa la Q-network para seleccionar acción,
    pero la target network para evaluar su valor.
    
    Cambio en el target:
    - DQN: y = r + γ max_a' Q(s', a'; θ⁻)
    - Double DQN: y = r + γ Q(s', argmax_a' Q(s', a'; θ); θ⁻)
    
    Objetivo: Modificar train_step() para implementar Double DQN.
    """
    
    def train_step(self) -> float:
        """
        TODO: Implementa la regla de actualización de Double DQN.
        
        Hint:
        1. Usa q_network para seleccionar argmax (mejor acción)
        2. Usa target_network para evaluar Q-value de esa acción
        """
        # TODO: Tu código aquí
        pass

def test_ejercicio_2():
    env_test = gym.make('CartPole-v1')
    double_agent = DoubleDQNAgent(
        state_dim=env_test.observation_space.shape[0],
        action_dim=env_test.action_space.n
    )
    double_agent.train(env_test, n_episodes=50, verbose=False)
    
    assert len(double_agent.training_history['reward']) > 0, "❌ El agente debe entrenarse"
    print("✅ ¡Excelente! Double DQN implementado.")
    print("   Double DQN reduce sobreestimación de Q-values.")
    env_test.close()
    return True

# test_ejercicio_2()  # Descomenta para probar

### 🔴 Ejercicio 3: Dueling DQN

Implementa Dueling DQN con arquitectura de ventaja separada.

In [ ]:
class DuelingQNetwork(nn.Module):
    """
    Dueling DQN architecture:
    Q(s,a) = V(s) + (A(s,a) - mean(A(s,:)))
    
    Objetivo: Implementar arquitectura con streams separados para V y A.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(DuelingQNetwork, self).__init__()
        
        # TODO: Implementa la arquitectura
        # Hint: Capa compartida → fork en dos streams (value y advantage)
        # Hint: Combina usando Q(s,a) = V(s) + A(s,a) - mean(A(s,:))
        pass
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        # TODO: Forward pass
        pass

def test_ejercicio_3():
    dueling_net = DuelingQNetwork(state_dim=4, action_dim=2)
    test_state = torch.randn(1, 4)
    output = dueling_net(test_state)
    
    assert output is not None, "❌ La red debe retornar output"
    assert output.shape == (1, 2), "❌ Output debe tener shape (1, 2)"
    print("✅ ¡Excelente! Dueling DQN implementado.")
    print("   Dueling ayuda cuando algunas acciones tienen poco impacto.")
    return True

# test_ejercicio_3()  # Descomenta para probar

## 📚 Resumen

### Conceptos Clave

- **DQN**: Combina Q-Learning con deep neural networks
- **Function Approximation**: Aproxima Q(s,a) con una red neuronal
- **Experience Replay**: Almacena transiciones y samplea mini-batches aleatorios
- **Target Network**: Red separada para calcular targets estables
- **Double DQN**: Reduce overestimation usando dos redes
- **Dueling DQN**: Separa value y advantage en la arquitectura

### Innovaciones de DQN

| Técnica | Problema que Resuelve | Cómo Funciona |
|---------|----------------------|---------------|
| **Experience Replay** | Correlación temporal | Sampleo aleatorio de transiciones pasadas |
| **Target Network** | Inestabilidad de targets | Red fija para targets, actualizada periódicamente |
| **Gradient Clipping** | Gradientes explosivos | Limita norma de gradientes |
| **Frame Stacking** | Capturar movimiento | Apilar últimos 4 frames |

### Comparación DQN Variants

| Algoritmo | Target | Ventaja Principal |
|-----------|--------|-------------------|
| **DQN** | $r + \gamma \max_{a'} Q(s', a'; \theta^-)$ | Base, simple |
| **Double DQN** | $r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta); \theta^-)$ | Menos overestimation |
| **Dueling DQN** | $V(s) + A(s,a)$ | Mejor con muchas acciones |

### Limitaciones de DQN

1. **Solo acciones discretas**: No funciona con acciones continuas
2. **Ineficiencia de muestras**: Requiere muchos episodios para converger
3. **Inestabilidad**: Puede divergir con malos hiperparámetros
4. **Off-policy**: No directamente optimiza la política que usa

### Lo que viene

DQN aprende una función de valor Q(s,a), pero:
- ¿Qué pasa si queremos aprender la política π(a|s) directamente?
- ¿Cómo manejar acciones continuas?

En el siguiente notebook veremos **Policy Gradients**, que optimiza directamente la política usando gradientes.

## 🔗 Recursos Adicionales

### 📄 Papers Fundamentales

- **"Playing Atari with Deep Reinforcement Learning"** - Mnih et al. (2013)
  - Paper original de DQN (arXiv)
  - Introduce experience replay y deep Q-learning

- **"Human-level control through deep reinforcement learning"** - Mnih et al. (2015)
  - Paper en Nature
  - Demuestra rendimiento humano en Atari

- **"Deep Reinforcement Learning with Double Q-learning"** - van Hasselt et al. (2015)
  - Introduce Double DQN
  - Reduce overestimation bias

- **"Dueling Network Architectures"** - Wang et al. (2016)
  - Dueling DQN architecture
  - Separa value y advantage

### 🎥 Videos

- **David Silver - Lecture 6: Value Function Approximation**
  - https://www.youtube.com/watch?v=UoPei5o4fps
  - Fundamentos de aproximación de funciones

### 💻 Implementaciones

- **DeepMind DQN**: https://github.com/deepmind/dqn
  - Implementación original en Lua/Torch

- **Stable-Baselines3**: https://stable-baselines3.readthedocs.io/
  - Implementaciones PyTorch optimizadas

## ➡️ Próximo Paso

En el siguiente notebook aprenderás sobre **Policy Gradients (REINFORCE)**, que optimiza directamente la política en lugar de la función de valor, permitiendo manejar acciones continuas.

**[Continuar con: 05. Policy Gradients →](05-policy-gradients.ipynb)**

---

<div align="center">
    
**¡Has dominado Deep Q-Learning, la base del Deep RL moderno! 🎉**

</div>